## Финальный проект (RAG для диалоговых систем)

В этом проекте мы создадим ассистента, который сможет отвечать на любые вопросы про жизнь известных личностей. Для этого мы реализуем поддержку диалога в RAG, а также к семантическому поиску по базе знаний мы добавим поиск информации в интернете. Поддержка диалога означает, что пользователь сможет уточнять любую информацию по предыдущему вопросу без необходимости задавать весь вопрос целиком.

### База знаний

База знаний состоит из первых абзацев русскоязычных статей из википедии про различных людей.

In [15]:
!pip freeze | grep langch

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


langchain==0.2.5
langchain-chroma==0.1.2
langchain-community==0.2.5
langchain-core==0.2.43
langchain-huggingface==0.0.3
langchain-text-splitters==0.2.4


In [1]:
with open('data/ru_wiki_person.txt', 'r') as f:
    articles = f.read().split('\n\n')

len(articles)

269086

In [2]:
from langchain_huggingface import HuggingFaceEmbeddings

#model_name instead of model!!
embeddings = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-large")

/Users/theo/karpov/nlp/.venv/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


In [4]:
from uuid import uuid4

from langchain_core.documents import Document
from langchain_chroma import Chroma


In [8]:
results = vector_store.similarity_search_with_score(
    "Кто создал фильм 'Берегись автомобиля'?",
    k=5,
)

In [9]:
results

[(Document(id='08548776-a38d-4596-990a-9117a3097077', metadata={}, page_content='Андре́й Арсе́ньевич Тарко́вский (4 апреля 1932, Завражье, Ивановская Промышленная область, СССР — 29 декабря 1986, Париж, Франция) — советский режиссёр театра и кино, сценарист; народный артист РСФСР (1980), лауреат Ленинской премии (1990 — "посмертно").Тарковский оказал значительное влияние на мировой кинематограф. Его фильмы «Андрей Рублёв» (1966), «Солярис» (1972), «Зеркало» (1974) и «Сталкер» (1979) периодически включаются в списки лучших кинопроизведений всех времён.Творчество Тарковского представляет собой значительное и необычное явление мировой культуры. Его фильмы образуют цикл о страданиях и надеждах человека, взявшего на себя бремя нравственной ответственности за весь мир. Концептуальные и художественные решения Тарковского отличаются оригинальностью и глубиной.'),
  0.47042006254196167),
 (Document(id='133327b1-2568-4eab-9fcd-971e860d1172', metadata={}, page_content='Чарльз Майкл (Чак) Пала́ник

In [6]:
import shutil

shutil.make_archive("chroma_db", 'zip', "chroma_db")

'/Users/theo/karpov/nlp/chroma_db.zip'

In [5]:
vector_store = Chroma(
    embedding_function=embeddings,
    persist_directory="./chroma_db",  # Where to save data locally, remove if not neccesary
)

# documents = []
# for i in range(100):
#   doc = Document(page_content=articles[i], id=i+1)
#   documents.append(doc)

documents = [Document(page_content=article) for article in articles[:100]]

uuids = [str(uuid4()) for _ in range(len(documents))]

vector_store.add_documents(documents=documents, ids=uuids)

['045d6137-69ee-4f63-9e76-5117d8de7a9f',
 'b3906eb5-7966-4568-91cf-f83bea9f24b6',
 'c0857905-d776-4d64-9b0a-28b68106a494',
 'cdd4acb2-4020-4883-8e0a-15ac5df05a8d',
 '9dd7828b-4336-41b6-b468-121cb976d92d',
 'dfdbf354-7820-4235-99aa-cd913d9cc0b2',
 'caf1ee4f-1ef7-42a6-96ba-ed3c45bb6405',
 '675ec056-573c-4e8c-8eaa-3b47718eb27c',
 '650e658f-9455-4037-8288-8e4689b7eac6',
 '913aab2b-7ede-41ad-b009-be86344473de',
 'fab6d339-d172-4e99-8d6b-6bfdc46df851',
 'd7e79ee2-d833-4cfe-a8fb-6ce2220b3288',
 '9f990d0f-c614-4b1d-98ae-b027b2a25e33',
 '9d41f518-448b-41c7-9368-1045e80c58e4',
 'b675a358-b89d-412f-b925-b77cc81f59c8',
 '6276e2ac-47cb-45c3-8d21-3c2561ec12ac',
 '05633b82-17a0-4e61-98e0-db3fa20ff44d',
 '2051a160-88f9-431f-b2a0-d89174a6d991',
 'e626f264-6c46-434f-8faf-b94cac72590e',
 'dde52547-4b86-4222-a6b1-d31d1bdc43fd',
 '7024b057-e924-407a-9a82-323115f3eb90',
 '7c1b2037-9f31-4299-9715-b584d61a310e',
 'eefa2c52-72fc-4b49-b2db-645eed2dd57a',
 '0f878672-ffc4-4cda-84ab-2380def236ab',
 '8d1403c6-ab7d-

In [2]:
articles[:5]

['Эльда́р Алекса́ндрович Ряза́нов (18 ноября 1927, Самара, СССР — 30 ноября 2015, Москва, Россия) — советский и российский кинорежиссёр, сценарист, актёр, поэт, драматург, телеведущий, педагог, продюсер; народный артист СССР (1984), лауреат Государственной премии СССР (1977) и Государственной премии РСФСР имени братьев Васильевых (1979).Среди шедевров советской киноклассики, созданных Эльдаром Рязановым, — комедии и мелодрамы «Карнавальная ночь» (1956), «Девушка без адреса» (1957), «Дайте жалобную книгу» (1965), «Берегись автомобиля» (1966), «Старики-разбойники» (1971), «Невероятные приключения итальянцев в России» (1973), «Ирония судьбы, или С лёгким паром» (1976), «Служебный роман» (1977), «Гараж» (1979), «О бедном гусаре замолвите слово» (1980), «Вокзал для двоих» (1982), «Жестокий романс» (1984), «Небеса обетованные» (1991).Рязанов — автор более 200 собственных телевизионных программ, с 1979 по 1985 год вёл телепередачу «Кинопанорама». Автор текста ряда широко популярных романсов, 

### Задание

В этом задании у вас будет гораздо больше свободы в реализации системы и не будет подсказок о том, как имплементировать те или иные компоненты. Вам предстоит самостоятельно организовать логику работы системы от начала до конца. Однако мы все же наметим план, которого стоит придерживаться:

1. Собрать векторную базу данных.
2. Написать движок для поиска текстов по базе данных.
3. Добавить функцию поиска текстов в интернете.
4. Добавить поддержку диалогового режима.
5. Составить из полученных компонент RAG и протестировать его работу.

Приступим! Ниже будет набор заданий с минимальной реализацией компонент, необходимых для RAG. Предполагается, для построения итоговой системы вы усложните данные компоненты по своему усмотрению.

__Задание 1.__ Создайте базу данных из __первых 100__ текстов в датасете. Вам предлагается использовать [ChromaDB](https://python.langchain.com/v0.2/docs/integrations/vectorstores/chroma/) из langchain. Она работает аналогично Qdrant, но, помимо всего прочего, ее проще сохранять на диск после создания. Это очень важно сделать, чтобы не считать эмбеддинги каждый раз заново.

Cохраните базу данных на диск с названием `chroma_db`. Никак не обрабатывайте тексты дополнительно (при построении RAG, вам, конечно, нужно будет резать тексты на куски). В грейдер сдайте zip архив с полученной базой данных ChromaDB. Мы будем загружать ее таким образом.
```
import zipfile

with zipfile.ZipFile('chroma_db.zip', 'r') as zip_ref:
    zip_ref.extractall('./')

db = Chroma(persist_directory="chroma_db", embedding_function=embedding_model)
```

Имя коллекции `collection_name` оставляйте в значении по умолчанию, иначе грейдер сломается. Как и раньше, в качестве модели эмбеддингов используйте `intfloat/multilingual-e5-large` из huggingface.

In [ ]:
import chromadb
import requests
import numpy as np
from transformers import AutoTokenizer, AutoModel
import torch
import os



### Retrieval Augmented Generation

Теперь можно собрать полную векторную базу данных и дописать вторую часть RAG – генерацию ответа. В качестве генеративной модели выберите `hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4` из `huggingface`. Это квантизованная версия Llama 3.1, которая отлично генерирует текст как на английском, так и на русском языке. Заметьте, что AWQ работает не на всех видеокартах. Например, такая квантизация не поддерживается на V100. Загрузить модель можно таким образом.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4")
model = AutoModelForCausalLM.from_pretrained("hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4")

__Задание 2.__ С помощью RAG сгенерируйте ответы к вопросам из файла `questions.txt`. Постарайтесь подобрать основной промпт таким образом, чтобы ответ был коротким и четким. Результат генерации сохраните в файл `answers.json` в виде списка ответов. 

```
import json

with open('answers.json', 'w', encoding='utf8') as f:
    json.dump(generated_answers, f, ensure_ascii=False)
```

In [ ]:
# ваш код здесь

### Поиск в интернете

Поиск в интернете можно использовать в том случае, если в базе знаний не нашлось достаточно подходящих текстов. Например, в Википедии ничего не написано про Александра Шабалина. Так что если вы спросите, кто является автором курса по NLP в karpov.courses, то без поиска в интернете, модель не сможет дать правильный ответ.

__Заданиe 3.__
Напишите функцию `internet_search`, которая принимает на вход текстовый запрос и аргумент `k` и возвращает набор из `k` текстов, найденных в интернете по полученному запросу. В качестве браузера проще всего использовать [`DuckDuckGO`](https://duckduckgo.com/) и специализированную [библиотеку](https://pypi.org/project/duckduckgo-search/) для него. Также скорее всего вам пригодятся библиотеки [`requests`](https://requests.readthedocs.io/en/latest/) и [`BeautifulSoup`](https://www.crummy.com/software/BeautifulSoup/bs4/doc/).

При встраивании этой компоненты в RAG подумайте о том, как понять, что релевантных текстов не оказалось в базе данных, а так же о том, какие тексты (куски?) и в каком количестве надо добавлять в контекст модели.

In [ ]:
# ваш код здесь

### Поддержка диалогов

Когда модель умеет отвечать на один поставленный вопрос - это хорошо. Но когда она умеет отвечать на уточняющие вопросы, учитывая историю общения – это еще лучше.

__Пример:__    
    – _Пользователь_: Кто был самым высоким человеком?   
    – _Ассистент_: Роберт Уодлоу.   
    – _Пользователь_: Какой у него был рост?   
    – _Ассистент_: 272 сантиметров.   

__Задание 4.__ Добавьте поддержку диалога в вашу систему RAG. С данной модификацией сгенерируйте ответы на вопросы
из файла `dialog_questions.txt` и запишите результат в файл `dialog_answers.json` в виде списка из пар ответов: ответ на первый вопрос и ответ на второй вопрос.  Если нужных документов нет в базе данных, используйте поиск в интернете.

_Подсказка:_ Для того, чтобы по новому вопросу можно было достать релевантные тексты из базы данных, вопрос нужно переформулировать, добавив нужную информацию из предыдущих сообщений пользователя. Поэтому при получении нового вопроса можно сделать запрос в LLM для уточнения запроса пользователя с учетом всей истории сообщений, а после этого искать релевантные тексты по уточненному запросу.

In [ ]:
# ваш код здесь

### Резюме

Ура! Теперь у вас есть ассистент, который с легкостью может заменить гугл. Если вы добавите к нему пользовательский интерфейс, то получите самый удобный способ поиска ответов на вопросы о людях. Это решение можно развивать и дальше, как улучшая имеющиеся компоненты, так и добавляя новые. Однако в рамках финального проекта мы остановимся на том, что есть.

Мы благодарим вас за прохождение данного курса и очень надеемся, что вы получили те знания, которые хотели, или даже больше. По крайней мере, теперь вы можете смело называть себя NLP-инженером :)